# NB_05 — SOURCE_05 Extraction v1.0

**Source:** *Highly-multiplexed microwave SQUID readout using the SLAC Microresonator Radio Frequency (SMuRF) Electronics for Future CMB and Sub-millimeter Surveys*

This notebook completes the `SOURCE_05` scaffold through the registered reusable extractor, validates the completed source record, writes the canonical YAML, and creates a downloadable export package.

Expected repository files:

```text
engineering_navigator/multiplexed_readout/source_records/
    SOURCE_05_smurf_multiplexed_readout.scaffold.yaml

tools/source_extractors/
    source_05.py
    registry.py
```


## 1. Configuration and repository discovery


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile

import yaml

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
REPO_ROOT_OVERRIDE = None

SOURCE_ID = "SOURCE_05"
SCAFFOLD_FILENAME = "SOURCE_05_smurf_multiplexed_readout.scaffold.yaml"
SOURCE_FILENAME = "SOURCE_05_smurf_multiplexed_readout.yaml"
PAPER_TITLE = (
    "Highly-multiplexed microwave SQUID readout using the "
    "SLAC Microresonator Radio Frequency (SMuRF) Electronics "
    "for Future CMB and Sub-millimeter Surveys"
)


def find_repo_root() -> Path:
    candidates = []

    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser().resolve())

    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])

    candidates.extend([
        Path("/content/sensors-becker"),
        Path("/home/dan/sensors-becker"),
        Path.home() / "sensors-becker",
    ])

    for candidate in candidates:
        if candidate.is_dir() and (candidate / "engineering_navigator").is_dir():
            return candidate

    if Path("/content").exists():
        target = Path("/content/sensors-becker")

        if target.exists():
            if (target / "engineering_navigator").is_dir():
                return target

            raise FileExistsError(
                f"{target} exists but does not look like sensors-becker."
            )

        subprocess.run(
            ["git", "clone", REPOSITORY_URL, str(target)],
            check=True,
        )

        if (target / "engineering_navigator").is_dir():
            return target

    raise FileNotFoundError(
        "Could not locate sensors-becker. Set REPO_ROOT_OVERRIDE explicitly."
    )


ROOT = find_repo_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SOURCE_DIR = (
    ROOT
    / "engineering_navigator"
    / "multiplexed_readout"
    / "source_records"
)
SCAFFOLD_PATH = SOURCE_DIR / SCAFFOLD_FILENAME
SOURCE_PATH = SOURCE_DIR / SOURCE_FILENAME

EXPORT_DIR = ROOT / "exports" / SOURCE_ID
EXPORT_ZIP = ROOT / "exports" / f"{SOURCE_ID}_export.zip"

print(f"Repository : {ROOT}")
print(f"Scaffold   : {SCAFFOLD_PATH.relative_to(ROOT)}")
print(f"Output     : {SOURCE_PATH.relative_to(ROOT)}")


## 2. Verify extractor registration


In [ ]:
from tools.source_extractors.registry import EXTRACTORS, extract_source

available = sorted(EXTRACTORS)
print("Registered extractors:", available)

if SOURCE_ID not in EXTRACTORS:
    raise ValueError(
        f"{SOURCE_ID} is not registered. Available: {available}"
    )

print("Registry validation: PASS")


## 3. Load and validate scaffold


In [ ]:
if not SCAFFOLD_PATH.exists():
    raise FileNotFoundError(
        f"Missing scaffold: {SCAFFOLD_PATH}\n"
        "Add SOURCE_05_smurf_multiplexed_readout.scaffold.yaml to the repo."
    )

scaffold = yaml.safe_load(
    SCAFFOLD_PATH.read_text(encoding="utf-8")
)

if not isinstance(scaffold, dict):
    raise TypeError(
        "SOURCE_05 scaffold must contain one top-level mapping."
    )

if scaffold.get("source_id") != SOURCE_ID:
    raise ValueError(
        f"Expected source_id {SOURCE_ID!r}; "
        f"found {scaffold.get('source_id')!r}"
    )

if scaffold.get("title") != PAPER_TITLE:
    print("WARNING: scaffold title differs from notebook PAPER_TITLE.")
    print("Scaffold:", scaffold.get("title"))
    print("Notebook:", PAPER_TITLE)

print("Loaded:", scaffold.get("title"))
print("Status:", scaffold.get("extraction_status"))


## 4. Run SOURCE_05 extractor


In [ ]:
completed_record = extract_source(SOURCE_ID, scaffold)

print(f"Authors                   : {len(completed_record.get('authors', []))}")
print(f"Materials                 : {len(completed_record.get('materials', []))}")
print(f"Fabrication methods       : {len(completed_record.get('fabrication_methods', []))}")
print(f"Design variables          : {len(completed_record.get('design_variables', []))}")
print(f"Reported values           : {len(completed_record.get('reported_values', []))}")
print(f"Measured outcomes         : {len(completed_record.get('measured_outcomes', []))}")
print(f"Equations                 : {len(completed_record.get('equations', []))}")
print(f"Engineering relationships : {len(completed_record.get('engineering_relationships', []))}")
print(f"Engineering constraints   : {len(completed_record.get('engineering_constraints', []))}")

gdt = completed_record.get("gdt_applicability", {})
print("GDT applicability status  :", gdt.get("status"))


## 5. Validate completed source record


In [ ]:
required_nonempty = [
    "authors",
    "materials",
    "design_variables",
    "reported_values",
    "measured_outcomes",
    "equations",
    "engineering_relationships",
    "engineering_constraints",
    "future_questions",
    "unreported_variables",
]

errors = []

if completed_record.get("source_id") != SOURCE_ID:
    errors.append(
        f"source_id mismatch: {completed_record.get('source_id')!r}"
    )

if completed_record.get("record_status") != "evidence_extracted":
    errors.append(
        f"unexpected record_status: "
        f"{completed_record.get('record_status')!r}"
    )

if not str(
    completed_record.get("extraction_status", "")
).startswith("complete"):
    errors.append(
        f"unexpected extraction_status: "
        f"{completed_record.get('extraction_status')!r}"
    )

for field in required_nonempty:
    value = completed_record.get(field)
    if not value:
        errors.append(f"{field} is empty or missing")

gdt = completed_record.get("gdt_applicability")
if not isinstance(gdt, dict):
    errors.append("gdt_applicability is missing or not a mapping")
else:
    if not gdt.get("status"):
        errors.append("gdt_applicability.status is missing")

# SOURCE_05 is intentionally expected to reject a direct GDT claim at this stage.
expected_gdt_status = (
    "discrete_structure_present_but_direct_GDT_not_established"
)

if isinstance(gdt, dict) and gdt.get("status") != expected_gdt_status:
    errors.append(
        "Unexpected GDT applicability status: "
        f"{gdt.get('status')!r}; expected {expected_gdt_status!r}"
    )

if errors:
    raise ValueError(
        "SOURCE_05 validation failed:\n- " + "\n- ".join(errors)
    )

print("Completed source-record validation: PASS")


## 6. Inspect GDT-relevant discrete structures


In [ ]:
gdt = completed_record["gdt_applicability"]

print("Status:")
print(" ", gdt.get("status"))

print("\nCandidate integer structures:")
for item in gdt.get("candidate_integer_structures", []):
    print(
        f"- {item.get('structure')}: "
        f"{item.get('evidence')} "
        f"→ {item.get('gdt_status')}"
    )

print("\nMissing direct GDT hypotheses:")
for item in gdt.get("missing_direct_hypotheses", []):
    print("-", item)

print("\nImportant negative result:")
print(gdt.get("important_negative_result"))


## 7. Inspect the strongest engineering constraints


In [ ]:
for item in completed_record.get("engineering_constraints", []):
    print(f"- {item.get('constraint')}: {item.get('statement')}")


## 8. Write canonical SOURCE_05 YAML


In [ ]:
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_PATH.write_text(
    yaml.safe_dump(
        completed_record,
        sort_keys=False,
        allow_unicode=True,
        width=110,
    ),
    encoding="utf-8",
)

# Round-trip validation.
roundtrip = yaml.safe_load(
    SOURCE_PATH.read_text(encoding="utf-8")
)

if roundtrip != completed_record:
    raise ValueError(
        "YAML round-trip changed the SOURCE_05 record."
    )

print("Wrote:", SOURCE_PATH.relative_to(ROOT))
print("YAML round-trip: PASS")


## 9. Build export package


In [ ]:
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

export_source = EXPORT_DIR / SOURCE_FILENAME
shutil.copy2(SOURCE_PATH, export_source)

manifest = {
    "source_id": SOURCE_ID,
    "title": completed_record.get("title"),
    "extraction_status": completed_record.get("extraction_status"),
    "files": [
        SOURCE_FILENAME,
    ],
    "counts": {
        "authors": len(completed_record.get("authors", [])),
        "materials": len(completed_record.get("materials", [])),
        "fabrication_methods": len(
            completed_record.get("fabrication_methods", [])
        ),
        "design_variables": len(
            completed_record.get("design_variables", [])
        ),
        "reported_values": len(
            completed_record.get("reported_values", [])
        ),
        "measured_outcomes": len(
            completed_record.get("measured_outcomes", [])
        ),
        "equations": len(
            completed_record.get("equations", [])
        ),
        "engineering_relationships": len(
            completed_record.get("engineering_relationships", [])
        ),
        "engineering_constraints": len(
            completed_record.get("engineering_constraints", [])
        ),
        "gdt_candidate_integer_structures": len(
            completed_record.get("gdt_applicability", {}).get(
                "candidate_integer_structures", []
            )
        ),
    },
    "gdt_applicability_status": (
        completed_record.get("gdt_applicability", {}).get("status")
    ),
}

manifest_path = EXPORT_DIR / "manifest.json"
manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

with zipfile.ZipFile(
    EXPORT_ZIP,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(EXPORT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print("Manifest:")
print(json.dumps(manifest, indent=2))
print()
print(f"Export package: {EXPORT_ZIP}")
print(f"Size: {EXPORT_ZIP.stat().st_size:,} bytes")


## 10. Download export ZIP in Colab


In [ ]:
try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("Automatic download is available only in Google Colab.")


## Handoff

After this notebook passes:

```text
SOURCE_05 scaffold
        ↓
source_05.py
        ↓
registry
        ↓
NB_05_SOURCE_05_EXTRACTION
        ↓
SOURCE_05_smurf_multiplexed_readout.yaml
        ↓
multiplexed-readout Engineering Object
        ↓
GDT applicability synthesis
```

The completed SOURCE_05 record should preserve the important distinction:

- real discrete architecture is present;
- collision and band-edge constraints are physically meaningful;
- no direct residue-class + coprimality mapping has yet been established.

That makes SOURCE_05 a substantially stronger GDT candidate than the electroplating variables without overstating the theorem application.
